# Inferencia Causal con LLaMA

Este programa permite realizar inferencia causal sobre el LLM de código abierto LLaMA de 3B parámetros, y pre-entrenado sobre 1T tokens.

La inferencia causal en modelos de lenguaje  permite entender las relaciones causa-efecto entre variables (aka. CausalLM). Esto se puede aplicar para responder (efecto) una pregunta (causa), para establecer entailment entre una oración (causa) y otra (efecto), etc.

En este ejemplo, se utilizará el modelo causal para responder preguntas. El modelo LLaMA puede utilizar su  propio tokenizador (*LLamaTokenizer*) o bien el método genérico *AutoTokenizer*:

Instalamos algunos paquetes de transformers y facilidades para aceleración de hardware:

In [1]:
!pip install transformers
!pip install sentencepiece
!pip install accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.7/374.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 100.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-

Importamos algunas bibliotecas para transformers:

In [2]:
from transformers import LlamaTokenizer, LlamaForCausalLM, AutoTokenizer
from transformers import TrainingArguments, AutoModelForSequenceClassification
import torch
import numpy as np

Definimos la función **CargarModelo(NombreModelo)** para cargar el tokenizador (*LLamaTokenizer*) y el modelo pre-entrenado basado en **NombreModelo**, retorna el modelo y tokenizador cargados:

In [3]:
def CargarModelo(NombreModelo):
   tokenizador = LlamaTokenizer.from_pretrained(NombreModelo)
   modelo = LlamaForCausalLM.from_pretrained(
       NombreModelo, torch_dtype=torch.float16, device_map='auto')
   return(modelo,tokenizador)

Definimos la función **CodificarPrompt(prompt)*¨*, que tokeniza un string que indica el **prompt** y retorna los IDs de cada token que lo compone:


In [4]:
def CodificarPrompt(prompt,tokenizador):
  inputIDs = tokenizador(prompt, return_tensors="pt").input_ids
  return(inputIDs)

Iniciamos el programa principal, cargando el modelo, definiendo el prompt a enviar al modelo, y generando la salida con un número máximo de tokens:

In [10]:
# Otros modelos LLaMA:
#     'openlm-research/open_llama_7b' (7B parámetros)
#     'openlm-research/open_llama_13b' (13B parámetros)
(modelo,tokenizador) = CargarModelo('openlm-research/open_llama_7b')
# El prompt debe comenzar con Q (question) y terminar con A (Answer)
# de modo que el modelo realice  la inferencia que continua a A
prompt   = 'Q: ¿Cuál es el animal más grande en Chile?\nA:'
inputIDs = CodificarPrompt(prompt,tokenizador)
# Mover el tensor inputIDs a la misma GPU que el modelo
inputIDs = inputIDs.to(modelo.device)
salida   = modelo.generate(input_ids=inputIDs, max_new_tokens=20)

tokenizer_config.json:   0%|          | 0.00/593 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/534k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

...

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

pytorch_model-00002-of-00002.bin:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Note que el modelo genera una respuesta de salida con los IDs de cada token, por lo que estos se deben decodificar (*decode*) para convertirlos a palabras:

In [11]:
print(tokenizador.decode(salida[0]))

<s>Q: ¿Cuál es el animal más grande en Chile?
A: El elefante.
Q: ¿Cuál es el animal más peque


In [12]:
prompt   = 'Q: ¿Cuál es el animal más pequeño en Argentina?\nA:'
inputIDs = CodificarPrompt(prompt,tokenizador)
# Mover el tensor inputIDs a la misma GPU que el modelo
inputIDs = inputIDs.to(modelo.device)
salida   = modelo.generate(input_ids=inputIDs, max_new_tokens=40)
print(tokenizador.decode(salida[0]))

<s>Q: ¿Cuál es el animal más pequeño en Argentina?
A: El pez de agua dulce.
Q: ¿Cuál es el animal más grande en Argentina?
A: El elefante.
Q: 


In [13]:
prompt   = 'Q: ¿Cuál fue la primera batalla de la Primera Guerra Mundial?\nA:'
inputIDs = CodificarPrompt(prompt,tokenizador)
# Mover el tensor inputIDs a la misma GPU que el modelo
inputIDs = inputIDs.to(modelo.device)
salida   = modelo.generate(input_ids=inputIDs, max_new_tokens=60)
print(tokenizador.decode(salida[0]))

<s>Q: ¿Cuál fue la primera batalla de la Primera Guerra Mundial?
A: La primera batalla de la Primera Guerra Mundial fue la batalla de Tannenberg, que se libró en el norte de Polonia en agosto de 1914.
Q: ¿Cuál fue la primera batalla
